# 01 — Data Quality

Objetivo: validar la integridad del dataset antes de analizar resultados experimentales.

En este notebook **no evaluamos qué campaña funciona mejor**. Solo comprobamos estructura, tipos, faltantes, duplicados, categorías, rangos y coherencia lógica.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 1. Localizar y cargar el dataset

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

csv_files = sorted(RAW_DIR.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'No se encontró ningún CSV en {RAW_DIR}. Colocá el dataset en data/raw/.')

DATA_PATH = csv_files[0]
print(f'Dataset seleccionado: {DATA_PATH.name}')

df = pd.read_csv(DATA_PATH)
df.head()

## 2. Dimensiones y estructura

In [ ]:
print(f'Filas: {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]}')
print('\nColumnas:')
print(df.columns.tolist())

In [ ]:
df.info()

## 3. Columnas esperadas

In [ ]:
expected_columns = {
    'recency', 'history', 'segment', 'mens', 'womens',
    'zip_code', 'newbie', 'channel', 'visit', 'conversion', 'spend'
}

actual_columns = set(df.columns)
missing_expected = sorted(expected_columns - actual_columns)
unexpected_columns = sorted(actual_columns - expected_columns)

print('Columnas esperadas ausentes:', missing_expected)
print('Columnas adicionales:', unexpected_columns)

## 4. Valores faltantes

In [ ]:
missing = (
    df.isna()
      .sum()
      .to_frame('missing_n')
      .assign(missing_pct=lambda x: x['missing_n'] / len(df) * 100)
      .sort_values('missing_n', ascending=False)
)
missing

## 5. Duplicados

In [ ]:
duplicate_rows = df.duplicated().sum()
print(f'Filas duplicadas exactas: {duplicate_rows:,}')

if duplicate_rows:
    display(df[df.duplicated(keep=False)].head(20))

## 6. Resumen descriptivo general

In [ ]:
df.describe(include='all').T

## 7. Categorías observadas

In [ ]:
categorical_candidates = ['segment', 'zip_code', 'channel']

for col in categorical_candidates:
    if col in df.columns:
        print(f'\n--- {col} ---')
        print(df[col].value_counts(dropna=False))

## 8. Validación de variables binarias

In [ ]:
binary_columns = ['mens', 'womens', 'newbie', 'visit', 'conversion']

binary_check = {}
for col in binary_columns:
    if col in df.columns:
        values = sorted(df[col].dropna().unique().tolist())
        binary_check[col] = {
            'values': values,
            'valid_01': set(values).issubset({0, 1})
        }

pd.DataFrame(binary_check).T

## 9. Rangos numéricos y valores imposibles

In [ ]:
numeric_columns = [c for c in ['recency', 'history', 'spend'] if c in df.columns]
df[numeric_columns].agg(['min', 'max', 'mean', 'median']).T

In [ ]:
quality_flags = {}

if 'history' in df.columns:
    quality_flags['negative_history'] = int((df['history'] < 0).sum())

if 'spend' in df.columns:
    quality_flags['negative_spend'] = int((df['spend'] < 0).sum())

if 'recency' in df.columns:
    quality_flags['nonpositive_recency'] = int((df['recency'] <= 0).sum())

pd.Series(quality_flags, name='count')

## 10. Coherencia entre conversión y gasto

In [ ]:
if {'conversion', 'spend'}.issubset(df.columns):
    conversion_spend_check = pd.Series({
        'conversion_0_spend_gt_0': ((df['conversion'] == 0) & (df['spend'] > 0)).sum(),
        'conversion_1_spend_le_0': ((df['conversion'] == 1) & (df['spend'] <= 0)).sum(),
    })
    display(conversion_spend_check.to_frame('count'))

## 11. Tamaño de los grupos experimentales

Esto se revisa solo como control estructural. El balance causal se analizará en el siguiente notebook.

In [ ]:
if 'segment' in df.columns:
    group_sizes = (
        df['segment']
          .value_counts(dropna=False)
          .rename('n')
          .to_frame()
          .assign(pct=lambda x: x['n'] / len(df) * 100)
    )
    display(group_sizes)

## 12. Data dictionary inicial

In [ ]:
data_dictionary = pd.DataFrame({
    'variable': ['recency', 'history', 'segment', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'visit', 'conversion', 'spend'],
    'role': [
        'pre-treatment', 'pre-treatment', 'treatment', 'pre-treatment', 'pre-treatment',
        'pre-treatment', 'pre-treatment', 'pre-treatment', 'outcome', 'outcome', 'outcome'
    ],
    'description': [
        'Recencia de compra previa',
        'Valor histórico de compras',
        'Grupo experimental asignado',
        'Indicador de compras previas de productos para hombres',
        'Indicador de compras previas de productos para mujeres',
        'Segmentación geográfica',
        'Indicador de cliente nuevo',
        'Canal histórico del cliente',
        'Visitó el sitio después del tratamiento',
        'Realizó una compra después del tratamiento',
        'Gasto posterior al tratamiento'
    ]
})
data_dictionary

## 13. Resumen de QA

In [ ]:
qa_summary = {
    'rows': len(df),
    'columns': df.shape[1],
    'missing_cells': int(df.isna().sum().sum()),
    'duplicate_rows': int(df.duplicated().sum()),
    'missing_expected_columns': missing_expected,
    'unexpected_columns': unexpected_columns,
}

qa_summary

## Criterio para cerrar esta etapa

Antes de pasar al análisis causal debemos poder responder:

- ¿El dataset tiene aproximadamente 64.000 observaciones?
- ¿Están presentes todas las variables esperadas?
- ¿Existen faltantes o duplicados relevantes?
- ¿Las variables binarias contienen únicamente 0/1?
- ¿Hay valores negativos o imposibles?
- ¿`conversion` y `spend` son lógicamente consistentes?
- ¿Los tres grupos experimentales están presentes?

El siguiente notebook será `02_randomization_check.ipynb`, donde evaluaremos el balance de covariables **pre-tratamiento** entre los tres grupos.